In [1]:
import numpy as np
import pandas as pd 

from scipy import stats

In [2]:
path="/home/dexter/Documents/GitHub/3D-textures/assets/experimental_corrected_results.csv"
df = pd.read_csv(path)
df.head()

,Printer,Texture,Test_Type,Comparison,Raw_Avg_Distance,Std_Deviation,Texture_Method_Error,Method_std_Error,Corrected_Distance
0,E,2,T,1 vs 2,0.000241,0.000202,0.000195,0.0,0.000046
1,E,2,T,1 vs 3,0.000249,0.000229,0.000195,0.0,0.000054
2,E,2,T,1 vs 4,0.000349,0.000325,0.000195,0.0,0.000154
3,E,2,T,1 vs 5,0.000365,0.000286,0.000195,0.0,0.000170
4,E,2,T,1 vs 6,0.000373,0.000356,0.000195,0.0,0.000178


In [4]:

n_raw = 5
n_method = 5

se_raw = df["Std_Deviation"] / np.sqrt(n_raw)
se_method = df["Method_std_Error"] / np.sqrt(n_method)
df["SE_Corrected"] = np.sqrt(se_raw**2 + se_method**2)

# Welch's t-statistic, per row, from summary stats
df["t_Score"] = (df["Raw_Avg_Distance"] - df["Texture_Method_Error"]) / df["SE_Corrected"]

# Welch–Satterthwaite degrees of freedom, per row
df["df_welch"] = (se_raw**2 + se_method**2)**2 / (
    (se_raw**4) / (n_raw - 1) + (se_method**4) / (n_method - 1)
)

# One-tailed p-value: testing whether raw distance significantly exceeds method error
df["p_value"] = 1 - stats.t.cdf(df["t_Score"], df["df_welch"])

df["Corrected_Distance_um"] = df["Corrected_Distance"] * 1e6
df["SE_Corrected_um"] = df["SE_Corrected"] * 1e6

df["Formatted_Result"] = df.apply(
    lambda r: f"{r['Corrected_Distance_um']:.2f} ± {r['SE_Corrected_um']:.2f} μm",
    axis=1,
)

print(df[["Printer", "Comparison", "Formatted_Result", "t_Score", "p_value"]])

    Printer Comparison    Formatted_Result   t_Score   p_value
0         E     1 vs 2    45.74 ± 90.37 μm  0.506150  0.319687
1         E     1 vs 3   53.54 ± 102.30 μm  0.523339  0.314196
2         E     1 vs 4  154.40 ± 145.46 μm  1.061455  0.174160
3         E     1 vs 5  170.43 ± 128.04 μm  1.331050  0.126986
4         E     1 vs 6  177.83 ± 159.22 μm  1.116872  0.163303
..      ...        ...                 ...       ...       ...
447       B     6 vs 1  231.19 ± 146.90 μm  1.573784  0.095325
448       B     6 vs 2    97.28 ± 88.81 μm  1.095346  0.167446
449       B     6 vs 3    96.14 ± 96.16 μm  0.999793  0.186995
450       B     6 vs 4  202.22 ± 159.05 μm  1.271412  0.136241
451       B     6 vs 5    76.76 ± 91.71 μm  0.836944  0.224856

[452 rows x 5 columns]


In [5]:
# 1. Calculate Mean Distance and Grand SE per printer
printer_summary = (
    df.groupby("Printer")
    .agg(
        Mean_Distance_um=("Corrected_Distance", lambda x: x.mean() * 1e6),
        Total_Tests=("p_value", "count"),
        # Count how many of the rows actually achieved statistical significance
        Significant_Tests=("p_value", lambda x: (x < 0.05).sum()),
    )
    .reset_index()
)

# 2. Compute the true Grand SE across the printer groups
sum_sq_se = (
    df.groupby("Printer")["SE_Corrected"]
    .apply(lambda x: np.sum(x**2))
    .reset_index()
)
printer_summary = printer_summary.merge(sum_sq_se, on="Printer")
printer_summary["Grand_SE_um"] = (
    np.sqrt(printer_summary["SE_Corrected"]) / printer_summary["Total_Tests"]
) * 1e6

# 3. Calculate the percentage of tests that broke through the noise floor
printer_summary["Significance_Rate"] = (
    printer_summary["Significant_Tests"] / printer_summary["Total_Tests"]
) * 100



# 4. Clean up formatting
printer_summary["Averaged_Geometry"] = printer_summary.apply(
    lambda r: f"{r['Mean_Distance_um']:.2f} ± {r['Grand_SE_um']:.2f} μm", axis=1
)
printer_summary["Success_Ratio"] = printer_summary.apply(
    lambda r: f"{int(r['Significant_Tests'])} / {int(r['Total_Tests'])} ({r['Significance_Rate']:.1f}%)",
    axis=1,
)

print(printer_summary[["Printer", "Total_Tests", "Averaged_Geometry", "Success_Ratio"]])

  Printer  Total_Tests  Averaged_Geometry     Success_Ratio
0       B          180    69.93 ± 7.53 μm    0 / 180 (0.0%)
1       E          160  259.91 ± 18.92 μm  16 / 160 (10.0%)
2       R          112   94.87 ± 10.40 μm    7 / 112 (6.2%)
